# Open-weight scale ladder — judging run

Reproduces the judging step of the paper with open weights. Everything downstream of the labels
is deterministic, so this notebook is the only part that needs a GPU.

**Protocol is fixed in `scale_prereg.py` and this notebook does not change it.** Decoding is
temperature 0 / top_p 1 / max_tokens 1024 / seed 20260908, the strict parser is the only parser,
and Qwen3 thinking mode is disabled. Every run writes a manifest recording the resolved
HuggingFace commit sha, quantisation and serving-stack versions.

Runtime: **A100 or L4**. Set it under Runtime -> Change runtime type before running anything.

In [ ]:
# 1 · repository and dependencies
REPO = "https://github.com/Taekyoon/academic_ir_research_2026.git"

import os, subprocess, sys

# Clone, or reuse an existing checkout, and FAIL LOUDLY otherwise. An earlier version of this
# cell wrote `git clone ... 2>/dev/null || echo "already cloned"`, which reported success for a
# genuine failure and then broke confusingly on the next line.
if os.path.basename(os.getcwd()) == "work":
    print("already inside the checkout:", os.getcwd())
elif os.path.isdir("work/.git"):
    os.chdir("work"); print("reusing existing checkout:", os.getcwd())
else:
    r = subprocess.run(["git", "clone", "--depth", "1", REPO, "work"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(f"git clone failed ({r.returncode}):\n{r.stderr.strip()}")
    os.chdir("work"); print("cloned to:", os.getcwd())

for p in ("code/scale_judge.py", "data/panel_sample.csv", "prereg/scale_prereg.py"):
    assert os.path.exists(p), f"checkout is incomplete: {p} missing"
print("HEAD:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip())

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# vLLM pulls its own torch; pin it so the manifest is meaningful across reruns
!pip -q install "vllm==0.11.0" "huggingface_hub>=0.26"
import vllm, torch
print("vllm", vllm.__version__, "| torch", torch.__version__)
print("bf16 supported:", torch.cuda.is_bf16_supported())

## 2 · Abstracts

The repository ships PMIDs and expert labels, not abstract text — the CLEF collection itself
ships only PMIDs, and this keeps publisher-copyrighted abstracts out of the repository. Fetch
them once; the file is cached for the rest of the session.

`--email` is optional. NCBI asks automated callers to identify themselves; supply your own
address or omit the flag.

In [ ]:
!python code/fetch_abstracts.py --pmids data/pmids_panel.txt --out clef_abstracts.jsonl
!wc -l clef_abstracts.jsonl

# Expect about 2,017 records, of which roughly 7% carry a title and no abstract body. Those are
# judged on the title alone and the judging script flags them title_only.
import json
recs = [json.loads(l) for l in open("clef_abstracts.jsonl") if l.strip()]
empty = sum(1 for r in recs if not (r.get("abstract") or "").strip())
print(f"records {len(recs):,} | title-only {empty:,} ({empty/max(len(recs),1):.1%})")

## 3 · Smoke test before spending the GPU

64 rows on the smallest arm. Check three things in the output: `strict` is 64/64, `out_tok med`
is small (a large median means a reasoning preamble is leaking through and thinking mode is not
actually off), and `rows/s` is high enough that the full run is affordable.

In [ ]:
import os
# The abstracts file is built by the cell above and is not shipped in the repository. Check it
# here so the smoke test cannot be run out of order.
assert os.path.exists("clef_abstracts.jsonl"), (
    "clef_abstracts.jsonl is missing - run the fetch cell above first")
n = sum(1 for _ in open("clef_abstracts.jsonl"))
print(f"abstracts on disk: {n:,} records")
assert n > 1900, f"only {n} records - the fetch looks incomplete, re-run the cell above"

!python code/scale_judge.py --arm qwen3-4b --condition A --limit 64 --outdir smoke
!cat smoke/scale_manifest_qwen3-4b_A.json

## 4 · The dense ladder, both conditions

4B -> 8B -> 14B -> 32B, conditions A and C. 2,025 pairs each.

**Precision is bf16 for every arm and there is no quantisation option** (pre-registration,
DISCLOSED). That removes precision as a confound by construction, and it makes the reachable
ladder a property of the device: 4B and 8B on 24 GB, up to 14B on 40 GB, all four only on 80 GB.
An arm that does not fit is refused before the model loads and is reported as **not attempted** —
it is never substituted by a quantised run at the same nominal size.

In [ ]:
ARMS = ["qwen3-4b", "qwen3-8b", "qwen3-14b", "qwen3-32b"]

# Precision is bf16 for every arm and there is no quantisation option. An arm that does not fit
# is refused by the script before it loads anything, and is reported as not attempted. Check what
# this session actually gave you before starting - 32B in bf16 needs an 80 GB card.
import torch, subprocess
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"{torch.cuda.get_device_name(0)} | {gb:.0f} GB")
print("reachable in bf16:", [a for a, need in
      [("qwen3-4b", 8.0), ("qwen3-8b", 16.4), ("qwen3-14b", 29.6), ("qwen3-32b", 65.6)]
      if need <= gb * 0.90])

for arm in ARMS:
    for cond in ("A", "C"):
        print(f"\n===== {arm} condition {cond}", flush=True)
        r = subprocess.run(["python", "code/scale_judge.py", "--arm", arm, "--condition", cond])
        if r.returncode != 0:
            print(f"  {arm} did not run - reported as not attempted, NOT quantised", flush=True)

## 5 · Lineage control at 8B

One non-Qwen point so an 8B result cannot be read as a Qwen property. Registered as descriptive
— a single control cannot separate lineage from scale.

Llama needs a gated-repo token; skip this cell if you do not have one and the control will be
reported as not run.

In [ ]:
from huggingface_hub import login
# login()   # uncomment and paste a token with access to meta-llama

for cond in ("A", "C"):
    !python code/scale_judge.py --arm llama-3.1-8b-instruct --condition $cond

## 6 · Collect

Zip the labels and manifests and download. The analysis runs off these two file types alone and
needs no GPU.

**If any arm printed a strict-parse rate below 0.95**, do not analyse it yet — the
pre-registration (R2) requires re-running that arm with `--guided` AND running one arm that
passed with `--guided` too, so the harness effect can be bounded. Free and guided labels are
never mixed within an arm.

In [ ]:
!zip -qr scale_labels.zip labels/
!ls -la labels/ | head -30
from google.colab import files
files.download("scale_labels.zip")